In [ ]:
from google.colab import files
uploaded = files.upload()   # izaberi DataSet.csv

Saving DataSet.csv to DataSet (1).csv


In [ ]:
import pandas as pd

df = pd.read_csv("DataSet.csv")

# Labela je 't'/'f' -> pretvaramo u 1/0
df["fraudulent"] = df["fraudulent"].map({"t": 1, "f": 0})

print("Oblik:", df.shape)
print("\nRaspodela klasa (0=pravi, 1=lažni):")
print(df["fraudulent"].value_counts())
print("\nProcenat lažnih:", round(df["fraudulent"].mean() * 100, 2), "%")

df.head(3)

Oblik: (17880, 18)

Raspodela klasa (0=pravi, 1=lažni):
fraudulent
0    17014
1      866
Name: count, dtype: int64

Procenat lažnih: 4.84 %


,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,in_balanced_dataset
0,Marketing Intern,"US, NY, New York",Marketing,NaN,"<h3>We're Food52, and we've created a groundbr...","<p>Food52, a fast-growing, James Beard Award-w...",<ul>\r\n<li>Experience with content management...,NaN,f,t,f,Other,Internship,NaN,NaN,Marketing,0,f
1,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"<h3>90 Seconds, the worlds Cloud Video Product...",<p>Organised - Focused - Vibrant - Awesome!<br...,<p><b>What we expect from you:</b></p>\r\n<p>Y...,<h3><b>What you will get from us</b></h3>\r\n<...,f,t,f,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0,f
2,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,<h3></h3>\r\n<p>Valor Services provides Workfo...,"<p>Our client, located in Houston, is actively...",<ul>\r\n<li>Implement pre-commissioning and co...,NaN,f,t,f,NaN,NaN,NaN,NaN,NaN,0,f


In [ ]:
import re

# Skidamo HTML tagove i višak razmaka
def clean_html(x):
    x = re.sub(r"<[^>]+>", " ", str(x))   # ukloni <tagove>
    x = re.sub(r"\s+", " ", x)            # sažmi razmake
    return x.strip()

text_cols = ["title", "company_profile", "description", "requirements", "benefits"]
df["text"] = df[text_cols].fillna("").agg(" ".join, axis=1).apply(clean_html)

# Dužina teksta po klasi — često se lažni i pravi razlikuju već po dužini
df["text_len"] = df["text"].str.len()
print("Prosečna dužina teksta po klasi:")
print(df.groupby("fraudulent")["text_len"].mean().round(0))

# Nedostajuće vrednosti
print("\nNedostajuće (%):")
print((df.isnull().mean() * 100).round(1).sort_values(ascending=False))

# Pogledaj jedan lažan oglas
print("\n--- Primer LAŽNOG oglasa ---")
print(df[df["fraudulent"] == 1]["text"].iloc[0][:800])

Prosečna dužina teksta po klasi:
fraudulent
0    2703.0
1    2081.0
Name: text_len, dtype: float64

Nedostajuće (%):
salary_range           84.0
department             64.6
required_education     45.3
benefits               40.2
required_experience    39.4
function               36.1
industry               27.4
employment_type        19.4
company_profile        18.5
requirements           15.0
location                1.9
title                   0.0
has_questions           0.0
has_company_logo        0.0
telecommuting           0.0
description             0.0
fraudulent              0.0
in_balanced_dataset     0.0
text                    0.0
text_len                0.0
dtype: float64

--- Primer LAŽNOG oglasa ---
IC&E Technician Staffing &amp; Recruiting done right for the Oil &amp; Energy Industry! Represented candidates are automatically granted the following perks: Expert negotiations on your behalf, maximizing your compensation package and implimenting ongoing increases Significant 

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             average_precision_score, roc_auc_score)

X = df["text"]
y = df["fraudulent"]

# stratify=y -> čuva isti odnos klasa u train i test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF: pretvara tekst u brojeve. unigrami+bigrami, izbacujemo retke reči (min_df=5)
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                             stop_words="english", min_df=5)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

# class_weight="balanced" -> model kažnjava greške na retkoj klasi jače (rešava imbalans)
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train_tfidf, y_train)

y_pred  = clf.predict(X_test_tfidf)
y_proba = clf.predict_proba(X_test_tfidf)[:, 1]

print(classification_report(y_test, y_pred, digits=3))
print("Matrica konfuzije  [[TN FP] [FN TP]]:")
print(confusion_matrix(y_test, y_pred))
print("\nPR-AUC (average precision):", round(average_precision_score(y_test, y_proba), 3))
print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3))

              precision    recall  f1-score   support

           0      0.995     0.986     0.991      3403
           1      0.766     0.908     0.831       173

    accuracy                          0.982      3576
   macro avg      0.881     0.947     0.911      3576
weighted avg      0.984     0.982     0.983      3576

Matrica konfuzije  [[TN FP] [FN TP]]:
[[3355   48]
 [  16  157]]

PR-AUC (average precision): 0.921
ROC-AUC: 0.988


In [ ]:
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())
coefs = clf.coef_[0]

top_fake = np.argsort(coefs)[-25:][::-1]      # najveći pozitivni -> ka "lažno"
top_real = np.argsort(coefs)[:25]             # najveći negativni -> ka "pravo"

print("🚩 Reči koje najviše ukazuju na LAŽAN oglas:")
for i in top_fake:
    print(f"   {feature_names[i]:30s} {coefs[i]:+.2f}")

print("\n✅ Reči koje najviše ukazuju na PRAVI oglas:")
for i in top_real:
    print(f"   {feature_names[i]:30s} {coefs[i]:+.2f}")

🚩 Reči koje najviše ukazuju na LAŽAN oglas:
   link                           +4.03
   accion                         +3.64
   earn                           +3.43
   data entry                     +3.43
   money                          +3.11
   clerk                          +2.90
   subsea                         +2.88
   assistant                      +2.87
   cash                           +2.86
   hospital                       +2.84
   novation                       +2.83
   aecom                          +2.77
   surgical                       +2.68
   aptitude staffing              +2.67
   receptionist                   +2.62
   work home                      +2.60
   offshore                       +2.54
   entry                          +2.51
   accountant                     +2.40
   administrative assistant       +2.40
   000                            +2.37
   requirements                   +2.37
   aptitude                       +2.36
   engineering                    +2

In [ ]:
!pip install -q -U "datasets>=2.19" "pyarrow>=15.0" "transformers>=4.40" accelerate

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# Isti random_state i stratify kao u baseline-u -> isti test skup -> pošteno poređenje
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["fraudulent"], test_size=0.2, random_state=42, stratify=df["fraudulent"]
)

train_ds = Dataset.from_pandas(pd.DataFrame({"text": X_train.tolist(), "labels": y_train.tolist()}))
test_ds  = Dataset.from_pandas(pd.DataFrame({"text": X_test.tolist(),  "labels": y_test.tolist()}))

# max_length=256 zbog brzine (oglasi su dugi; 512 hvata više ali je sporije)
def tok(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_ds = train_ds.map(tok, batched=True)
test_ds  = test_ds.map(tok, batched=True)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
print("Spremno. Train:", len(train_ds), "Test:", len(test_ds))

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/14304 [00:00<?, ? examples/s]

Map:   0%|          | 0/3576 [00:00<?, ? examples/s]

Spremno. Train: 14304 Test: 3576


In [ ]:
import torch
import numpy as np
import torch.nn as nn
from scipy.special import softmax
from sklearn.metrics import precision_recall_fscore_support, average_precision_score
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

# Težine klasa -> model jače kažnjava greške na retkoj "lažno" klasi
n0, n1 = (y_train == 0).sum(), (y_train == 1).sum()
N = len(y_train)
class_weights = torch.tensor([N/(2*n0), N/(2*n1)], dtype=torch.float)
print("Težine klasa [pravo, lažno]:", class_weights.tolist())

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=1)[:, 1]
    preds = logits.argmax(-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average=None, labels=[0, 1])
    return {"precision_fake": p[1], "recall_fake": r[1], "f1_fake": f1[1],
            "pr_auc": average_precision_score(labels, probs)}

args = TrainingArguments(
    output_dir="out", num_train_epochs=3,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, eval_strategy="epoch", logging_steps=50, report_to="none",
)

trainer = WeightedTrainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
    data_collator=collator, compute_metrics=compute_metrics,
)
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Težine klasa [pravo, lažno]: [0.5254573225975037, 10.320345878601074]


Epoch,Training Loss,Validation Loss,Precision Fake,Recall Fake,F1 Fake,Pr Auc
1,0.268559,0.331646,0.907285,0.791908,0.845679,0.895986
2,0.348259,0.386870,0.953020,0.820809,0.881988,0.916597
3,0.101193,0.374265,0.936709,0.855491,0.894260,0.925663


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2682, training_loss=0.26094931172933444, metrics={'train_runtime': 1142.7106, 'train_samples_per_second': 37.553, 'train_steps_per_second': 2.347, 'total_flos': 2842220505563136.0, 'train_loss': 0.26094931172933444, 'epoch': 3.0})

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred = trainer.predict(test_ds)
y_pred = pred.predictions.argmax(-1)

print(classification_report(y_test, y_pred, digits=3))
print("Matrica konfuzije [[TN FP] [FN TP]]:")
print(confusion_matrix(y_test, y_pred))
print("\n--- Poređenje na klasi 'lažno' ---")
print("Baseline (TF-IDF):  precision 0.766 | recall 0.908 | PR-AUC 0.921")
print(f"DistilBERT:         precision {pred.metrics['test_precision_fake']:.3f} | "
      f"recall {pred.metrics['test_recall_fake']:.3f} | PR-AUC {pred.metrics['test_pr_auc']:.3f}")

              precision    recall  f1-score   support

           0      0.993     0.997     0.995      3403
           1      0.937     0.855     0.894       173

    accuracy                          0.990      3576
   macro avg      0.965     0.926     0.945      3576
weighted avg      0.990     0.990     0.990      3576

Matrica konfuzije [[TN FP] [FN TP]]:
[[3393   10]
 [  25  148]]

--- Poređenje na klasi 'lažno' ---
Baseline (TF-IDF):  precision 0.766 | recall 0.908 | PR-AUC 0.921
DistilBERT:         precision 0.937 | recall 0.855 | PR-AUC 0.926


In [ ]:
!pip install -q groq

from getpass import getpass
import os
os.environ["GROQ_API_KEY"] = getpass("Nalepi svoj Groq API ključ: ")

from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Groq klijent spreman.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 9.1 MB/s eta 0:00:00
Nalepi svoj Groq API ključ: ··········
Groq klijent spreman.


In [ ]:
import torch
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

def bert_predict(text):
    """Vrati verovatnoću da je oglas LAŽAN (0-1)."""
    inputs = tokenizer(text, truncation=True, max_length=256, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    prob_fake = F.softmax(logits, dim=1)[0, 1].item()
    return prob_fake

In [ ]:
import json

def llm_explain(text, prob_fake):
    """LLM analizira oglas i vraća objašnjenje crvenih signala (na srpskom)."""
    system = (
        "Ti si analitičar za detekciju prevara u oglasima za posao. "
        "Analiziraš tekst oglasa i identifikuješ konkretne 'crvene signale' koji "
        "ukazuju na moguću prevaru (nerealna zarada, hitnost, traženje uplate ili "
        "ličnih podataka, uopšten/šturi opis, sumnjiv kontakt, obećanja bez pokrića). "
        "Odgovaraš ISKLJUČIVO na srpskom jeziku i ISKLJUČIVO validnim JSON objektom, "
        "bez ikakvog teksta pre ili posle."
    )
    user = f"""Model mašinskog učenja je ovom oglasu dao verovatnoću prevare od {prob_fake:.0%}.

Tekst oglasa:
\"\"\"{text[:3000]}\"\"\"

Vrati JSON tačno u ovom formatu:
{{
  "procena": "verovatno lažno" | "sumnjivo" | "verovatno pravo",
  "crveni_signali": ["kratak signal 1", "signal 2", "..."],
  "objasnjenje": "2-3 rečenice zašto, ljudskim jezikom"
}}"""

    resp = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user}],
        temperature=0.2,
        response_format={"type": "json_object"},  # tera model da vrati validan JSON
    )
    return json.loads(resp.choices[0].message.content)

In [ ]:
def analyze_ad(text):
    prob = bert_predict(text)
    explanation = llm_explain(text, prob)
    return {"verovatnoca_prevare": round(prob, 3), **explanation}

# Uzmi jedan stvarno lažan oglas iz test skupa
primer = df[df["fraudulent"] == 1]["text"].iloc[3]

rezultat = analyze_ad(primer)
print("VEROVATNOĆA PREVARE (DistilBERT):", rezultat["verovatnoca_prevare"])
print("PROCENA (LLM):", rezultat["procena"])
print("\nCRVENI SIGNALI:")
for s in rezultat["crveni_signali"]:
    print("  🚩", s)
print("\nOBJAŠNJENJE:")
print(" ", rezultat["objasnjenje"])

VEROVATNOĆA PREVARE (DistilBERT): 0.694
PROCENA (LLM): verovatno lažno

CRVENI SIGNALI:
  🚩 Nedostatak detalja
  🚩 Ponavljanje iste fraze
  🚩 Nedostatak kontakt informacija

OBJAŠNJENJE:
  Oglas sadrži samo ponovljenu frazu bez ikakvih informacija o kompaniji, lokaciji ili uslovima. Takav nedostatak sadržaja i ponavljanje ukazuju na potencijalnu prevaru. Bez dodatnih podataka nije moguće potvrditi legitimnost ponude.


In [ ]:
# Sačuvaj DistilBERT (model + tokenizer)
save_dir = "fakead_model"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Spakuj u zip da lakše skineš iz Colab-a
import shutil
shutil.make_archive("fakead_model", "zip", save_dir)
print("Sačuvano u fakead_model/ i fakead_model.zip")

# Skini na računar (za lokalni Streamlit)
from google.colab import files
files.download("fakead_model.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sačuvano u fakead_model/ i fakead_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib, os

# Ponovo napravi baseline iz df (isti split kao ranije)
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["fraudulent"], test_size=0.2, random_state=42, stratify=df["fraudulent"]
)
vectorizer = TfidfVectorizer(max_features=20000, ngram_range=(1, 2),
                             stop_words="english", min_df=5)
Xtr = vectorizer.fit_transform(X_train)
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(Xtr, y_train)

# Snimi i PROVERI veličinu
joblib.dump({"vectorizer": vectorizer, "clf": clf}, "baseline.joblib")
size_kb = os.path.getsize("baseline.joblib") / 1024
print(f"Veličina fajla: {size_kb:.1f} KB")   # treba da bude nekoliko stotina KB, NE 0

Veličina fajla: 987.2 KB


In [ ]:
from google.colab import files
files.download("baseline.joblib")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>